<a href="https://colab.research.google.com/github/yaesur/business_python/blob/main/%EC%83%81_%ED%95%98%EB%B0%98%EA%B8%B0%EB%B3%80%EC%88%98.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

file_name = '커트라인 데이터.xlsx'
xls = pd.ExcelFile(file_name)
sheet_names = xls.sheet_names

# 가점이 있는 열 번호 (0부터 시작)
score_cols = [6, 6, 6, 6, 6, 6, 6, 5, 5, 8]
# 순위 열은 보통 가점 열 바로 왼쪽(-1)에 위치하므로 자동 계산
rank_cols = [col - 1 for col in score_cols]

all_data = []

for i, name in enumerate(sheet_names):
    df = pd.read_excel(file_name, sheet_name=name, skiprows=2)

    temp = pd.DataFrame()
    temp['자치구'] = df.iloc[:, 1]

    # 순위와 점수 열을 함께 가져옴
    temp['순위'] = df.iloc[:, rank_cols[i]]
    temp['점수'] = df.iloc[:, score_cols[i]]
    temp['시트명'] = name

    if name.endswith('-1'):
        temp['차수'] = '1차(상반기)'
    elif name.endswith('-2'):
        temp['차수'] = '2차(하반기)'

    all_data.append(temp)

# 기초 전처리
final_df = pd.concat(all_data).dropna(subset=['자치구'])

# 순위 전처리 (글자 '순위' 떼고 숫자로 변환)
final_df['순위'] = final_df['순위'].astype(str).str.replace('순위', '', regex=False).str.strip()
final_df['순위'] = pd.to_numeric(final_df['순위'], errors='coerce')

# 점수 전처리 (글자 '점' 떼고 숫자로 변환)
final_df['점수'] = pd.to_numeric(final_df['점수'].astype(str).str.replace('점', '', regex=False).str.strip(), errors='coerce')

# 결측치 제거
final_df = final_df.dropna(subset=['순위', '점수'])
final_df['자치구'] = final_df['자치구'].str.strip()


# ========================================================
# 순위별로 데이터셋을 쪼개서 T-검정 및 추이 분석 진행
# ========================================================

for rank in [1, 2]:
    print(f"\n==================================================")
    print(f"               🎯 {rank}순위 모집 단위 분석               ")
    print(f"==================================================")

    # 해당 순위 데이터만 필터링
    rank_df = final_df[final_df['순위'] == rank]

    if rank_df.empty:
        print(f"{rank}순위 데이터가 존재하지 않습니다.")
        continue

    # 1. 상/하반기 평균 커트라인 비교
    analysis = rank_df.groupby('차수').agg({
        '자치구': 'count',
        '점수': 'mean'
    }).rename(columns={'자치구': '공급건수', '점수': '평균커트라인'}).round(2)

    print(f"--- [ {rank}순위 ] 1, 2차 모집 비교 결과 ---")
    print(analysis)

    # 2. 연도별/차수별 커트라인 추이
    rank_df['연도'] = rank_df['시트명'].str.split('-').str[0]
    trend = rank_df.groupby(['연도', '차수'])['점수'].mean().unstack()
    print(f"\n--- [ {rank}순위 ] 연도별/차수별 커트라인 추이 ---")
    print(trend.round(2))

    # 3. T-검정 수행 (해당 순위 내에서 상반기 vs 하반기)
    group_1 = rank_df[rank_df['차수'] == '1차(상반기)']['점수']
    group_2 = rank_df[rank_df['차수'] == '2차(하반기)']['점수']

    if len(group_1) > 1 and len(group_2) > 1:
        t_stat, p_val = stats.ttest_ind(group_1, group_2)
        print(f"\n검정 결과 (p-value): {p_val:.10f}")

        if p_val < 0.05:
            print(f">>> 결론: {rank}순위 내에서 상/하반기 차이는 통계적으로 유의미함! (필수 변수)")
        else:
            print(f">>> 결론: {rank}순위 내에서 상/하반기 차이가 우연일 수 있음. (선택 변수)")
    else:
        print("\n검정 불가: 비교할 표본 수가 부족합니다.")


               🎯 1순위 모집 단위 분석               
--- [ 1순위 ] 1, 2차 모집 비교 결과 ---
         공급건수  평균커트라인
차수                   
1차(상반기)   312    6.39
2차(하반기)   347    5.54

--- [ 1순위 ] 연도별/차수별 커트라인 추이 ---
차수    1차(상반기)  2차(하반기)
연도                    
2021     6.54     4.94
2022     5.36     5.27
2023     6.05     5.24
2024     6.24     5.94
2025     7.38     5.91

검정 결과 (p-value): 0.0000046236
>>> 결론: 1순위 내에서 상/하반기 차이는 통계적으로 유의미함! (필수 변수)

               🎯 2순위 모집 단위 분석               
--- [ 2순위 ] 1, 2차 모집 비교 결과 ---
         공급건수  평균커트라인
차수                   
1차(상반기)   173    6.39
2차(하반기)   260    5.97

--- [ 2순위 ] 연도별/차수별 커트라인 추이 ---
차수    1차(상반기)  2차(하반기)
연도                    
2021     5.67     4.63
2022     5.90     5.77
2023     6.30     6.78
2024     7.15     7.32
2025     6.21     6.02

검정 결과 (p-value): 0.0116496767
>>> 결론: 2순위 내에서 상/하반기 차이는 통계적으로 유의미함! (필수 변수)


/tmp/ipykernel_2139/1253712487.py:76: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  rank_df['연도'] = rank_df['시트명'].str.split('-').str[0]
/tmp/ipykernel_2139/1253712487.py:76: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  rank_df['연도'] = rank_df['시트명'].str.split('-').str[0]


In [5]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

file_name = '커트라인 데이터.xlsx'
xls = pd.ExcelFile(file_name)
sheet_names = xls.sheet_names

score_cols = [6, 6, 6, 6, 6, 6, 6, 5, 5, 8]

all_data = []

for i, name in enumerate(sheet_names):

  df = pd.read_excel(file_name, sheet_name=name, skiprows=2)

  temp = pd.DataFrame()
  temp['자치구'] = df.iloc[:, 1]

  temp['점수'] = df.iloc[:, score_cols[i]]
  temp['시트명'] = name

  if name.endswith('-1'):
    temp['차수'] = '1차(상반기)'
  elif name.endswith('-2'):
    temp['차수'] = '2차(하반기)'

  all_data.append(temp)

final_df = pd.concat(all_data).dropna(subset=['자치구'])

final_df['점수'] = pd.to_numeric(final_df['점수'].astype(str).str.replace('점', ''), errors='coerce')
final_df = final_df.dropna(subset=['점수'])
final_df['자치구'] = final_df['자치구'].str.strip()

analysis = final_df.groupby('차수').agg({
  '자치구': 'count',
  '점수': 'mean'
}).rename(columns={'자치구': '공급건수', '점수': '평균커트라인'}).round(2)

print("--- 1, 2차 모집 비교 결과 ---")
print(analysis)

final_df['연도'] = final_df['시트명'].str.split('-').str[0]
trend = final_df.groupby(['연도', '차수'])['점수'].mean().unstack()

print("\n--- 연도별/차수별 커트라인 추이 ---")
print(trend.round(2))

from scipy import stats

group_1 = final_df[final_df['차수'] == '1차(상반기)']['점수']
group_2 = final_df[final_df['차수'] == '2차(하반기)']['점수']

# T-검정 실시 (두 집단의 평균 차이 확인)
t_stat, p_val = stats.ttest_ind(group_1, group_2)

print(f"검정 결과 (p-value): {p_val:.10f}")

if p_val < 0.05:
  print(">>> 결론: 상/하반기 차이는 통계적으로 유의미함! (필수 변수)")
  print("합격 예측을 할 때 모집 시점이 언제인지를 꼭 입력받아야 합니다.")
else:
  print(">>> 결론: 상/하반기 차이가 우연일 수 있음. (선택 변수)")
  print("굳이 시점을 나누지 않고 통합해서 예측해도 큰 차이가 없을 수 있습니다.")

--- 1, 2차 모집 비교 결과 ---
         공급건수  평균커트라인
차수                   
1차(상반기)   487    6.39
2차(하반기)   625    5.71

--- 연도별/차수별 커트라인 추이 ---
차수    1차(상반기)  2차(하반기)
연도                    
2021     6.45     4.81
2022     5.58     5.46
2023     6.13     5.67
2024     6.57     6.15
2025     6.94     5.99
검정 결과 (p-value): 0.0000001747
>>> 결론: 상/하반기 차이는 통계적으로 유의미함! (필수 변수)
합격 예측을 할 때 모집 시점이 언제인지를 꼭 입력받아야 합니다.
